Bibliotekos

In [1]:
import os

import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import kpss

Duomenų įsikėlimas

In [ ]:
data = pd.read_csv(os.path.join("dm_project_dataset_alpha.csv"), sep = ',')
data['time'] = pd.to_datetime(data['time'])
data.head(5)

,time,airTemperature,seaLevelPressure,relativeHumidity,xgb_precipitation,xgb_cloudCover,feelsLikeTemperature_fixed,windSpeed_fixed,windGust_fixed,windDirection_fixed,...,O3_Error,dust,sia,pm2p5,pans,nmvoc,nh3,pm10,tp,t2m
0,2018-07-01,11.295833,1010.637500,72.791667,9.100000e+00,89.375000,10.933333,3.762500,13.9,356.0,...,0.823016,0.011452,0.412340,1.127883,1.174715,2.292138,0.105974,1.338105,14.653358,12.970276
1,2018-07-02,10.866667,1006.662500,94.000000,1.190000e+01,91.500000,10.429167,2.516667,7.0,327.0,...,0.756530,0.011515,0.946260,1.609913,1.935448,3.792921,0.198428,1.752331,6.475957,14.779846
2,2018-07-03,13.387500,1008.454167,90.041667,1.000000e+01,89.375000,13.387500,2.537500,8.3,271.0,...,1.624589,0.044194,0.676988,1.534471,1.787800,2.332084,0.118859,1.789138,9.607317,15.965973
3,2018-07-04,13.591667,1009.237500,88.291667,1.390000e+01,88.333333,13.591667,2.358333,8.7,288.0,...,0.759276,0.128728,0.833161,3.229130,1.201320,2.113538,0.361157,4.323638,0.013308,18.015717
4,2018-07-05,16.291667,1008.079167,70.208333,2.220446e-16,66.000000,16.291667,1.283333,6.1,293.0,...,0.892097,0.097650,0.737482,2.768797,1.206750,2.267631,0.316533,3.718909,-0.000741,19.040314


In [4]:
data.columns

Index(['time', 'airTemperature', 'seaLevelPressure', 'relativeHumidity',
       'xgb_precipitation', 'xgb_cloudCover', 'feelsLikeTemperature_fixed',
       'windSpeed_fixed', 'windGust_fixed', 'windDirection_fixed',
       'absorbing_aerosol_index', 'NO2_column_number_density',
       'stratospheric_NO2_column_number_density',
       'NO2_slant_column_number_density', 'tropopause_pressure',
       'H2O_column_number_density', 'BrO', 'BrO_Error', 'NO2', 'NO2_Error',
       'O3', 'O3_Error', 'dust', 'sia', 'pm2p5', 'pans', 'nmvoc', 'nh3',
       'pm10', 'tp', 't2m'],
      dtype='object')

In [5]:
data_clean = data.drop(columns = ['BrO_Error', 'NO2_Error', 'O3_Error'])

Stacionarumas

In [6]:
def stationary_fun(timeseries):
    ts = timeseries.dropna()
    
    adf_result = adfuller(ts, autolag='AIC')
    adf_p = adf_result[1]
    
    kpss_stat, kpss_p, lags, crit = kpss(ts, regression='c')
    
    if adf_p < 0.05 and kpss_p > 0.05:
        return True
    else:
        return False

In [7]:
stationary_cols = []
non_stationary_cols = []

target_cols = [c for c in data_clean.columns if c != 'time']

for col in target_cols:
    if stationary_fun(data[col]):
        stationary_cols.append(col)
    else:
        non_stationary_cols.append(col)

C:\Users\eveli\AppData\Local\Temp\ipykernel_2880\220190549.py:7: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_stat, kpss_p, lags, crit = kpss(ts, regression='c')
C:\Users\eveli\AppData\Local\Temp\ipykernel_2880\220190549.py:7: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_stat, kpss_p, lags, crit = kpss(ts, regression='c')
C:\Users\eveli\AppData\Local\Temp\ipykernel_2880\220190549.py:7: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_stat, kpss_p, lags, crit = kpss(ts, regression='c')
C:\Users\eveli\AppData\Local\Temp\ipykernel_2880\220190549.py:7: InterpolationWarning: The test statistic is outside of the ran

In [8]:
print(f"Stacionarūs: {stationary_cols}")
print(f"Nestacionarūs: {non_stationary_cols}")

Stacionarūs: ['airTemperature', 'seaLevelPressure', 'relativeHumidity', 'xgb_precipitation', 'xgb_cloudCover', 'windSpeed_fixed', 'windGust_fixed', 'windDirection_fixed', 'NO2_column_number_density', 'stratospheric_NO2_column_number_density', 'NO2_slant_column_number_density', 'tropopause_pressure', 'H2O_column_number_density', 'BrO', 'O3', 'dust', 'sia', 'nh3', 'pm10', 'tp', 't2m']
Nestacionarūs: ['feelsLikeTemperature_fixed', 'absorbing_aerosol_index', 'NO2', 'pm2p5', 'pans', 'nmvoc']


In [9]:
# import seaborn as sns
# correlation_matrix = data.select_dtypes(include='number').corr(method = 'spearman')
# sns.heatmap(correlation_matrix)
# plt.show()

In [10]:
# correlation_matrix